## Install library yang dibutuhkan

In [1]:
!pip install librosa fastdtw numpy matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567859 sha256=f60781d3c77f8911ca6e353ae7c1c991a18da1219f1909aac54a519d2a371567
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [ ]:
import librosa
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
from google.colab import files
import io

# Konfigurasi
TARGET_SR = 4000  # Downsample ke 4kHz agar DTW cepat. Raw audio 44kHz terlalu berat.

def load_and_prep_raw_audio(uploaded_file_content):
    """
    Memuat raw audio, menghapus hening (trim), dan normalisasi amplitudo.
    """
    # Load audio dari byte content
    y, sr = librosa.load(io.BytesIO(uploaded_file_content), sr=TARGET_SR)

    # 1. Trim Silence (Hapus hening di awal & akhir agar fokus ke gelombang suara)
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)

    # 2. Normalisasi Amplitudo (Min-Max Scaling ke range -1 sampai 1)
    # Ini PENTING agar rekaman yang keras dan pelan bisa dibandingkan bentuknya.
    if np.max(np.abs(y_trimmed)) > 0:
        y_normalized = y_trimmed / np.max(np.abs(y_trimmed))
    else:
        y_normalized = y_trimmed

    return y_normalized

def calculate_dtw_distance(series1, series2):
    """
    Menghitung jarak DTW antara dua array raw audio.
    """
    distance, path = fastdtw(series1, series2, dist=euclidean)
    return distance

## Upload File

In [ ]:
print("=== UPLOAD FILE REFERENSI ===")
print("Silakan upload 1 file audio REFERENSI untuk 'BUKA'")
ref_buka_upload = files.upload()
ref_buka_name = next(iter(ref_buka_upload))
ref_buka_raw = load_and_prep_raw_audio(ref_buka_upload[ref_buka_name])
print(f"Referensi Buka '{ref_buka_name}' dimuat.")

print("\nSilakan upload 1 file audio REFERENSI untuk 'TUTUP'")
ref_tutup_upload = files.upload()
ref_tutup_name = next(iter(ref_tutup_upload))
ref_tutup_raw = load_and_prep_raw_audio(ref_tutup_upload[ref_tutup_name])
print(f"Referensi Tutup '{ref_tutup_name}' dimuat.")

Silakan upload file-file .wav Anda (Buka dan Tutup):


KeyboardInterrupt: 

In [9]:
import numpy as np
import pandas as pd
from google.colab import files
from scipy.io import wavfile
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import io

# =======================
# Konfigurasi
# =======================
TARGET_SR = 1000
SILENCE_THRESHOLD = 0.01


# =======================
# Fungsi FIX: Paksa audio selalu 1D & bersih
# =======================
def force_1d_clean(x):
    x = np.asarray(x, dtype=np.float32).flatten()
    x = x[np.isfinite(x)]        # hilangkan NaN / inf

    if x.size == 0:
        return np.array([0.0], dtype=np.float32)

    return x


# =======================
# Load WAV tanpa librosa
# =======================
def load_wav_raw(file_bytes):
    try:
        sr, data = wavfile.read(io.BytesIO(file_bytes))
    except Exception as e:
        print("❌ ERROR WAV:", e)
        return None, None

    # Stereo → ambil channel pertama
    if len(data.shape) > 1:
        data = data[:, 0]

    return sr, data.astype(np.float32)


# =======================
# Downsample manual
# =======================
def downsample(data, original_sr, target_sr):
    if original_sr == target_sr:
        return data

    factor = original_sr // target_sr
    if factor <= 0:
        return data

    return data[::factor]


# =======================
# Trim silence
# =======================
def trim_silence(x, threshold=SILENCE_THRESHOLD):
    if x.size == 0:
        return x

    mask = np.abs(x) > threshold

    if not np.any(mask):
        return np.array([0.0], dtype=np.float32)

    start = np.argmax(mask)
    end = len(mask) - np.argmax(mask[::-1])

    return x[start:end]


# =======================
# Normalisasi
# =======================
def normalize_audio(x):
    max_val = np.max(np.abs(x))
    return x / max_val if max_val > 0 else x


# =======================
# Fungsi utama proses audio
# =======================
def process_audio_no_librosa(file_bytes):
    sr, signal = load_wav_raw(file_bytes)
    if signal is None:
        return np.array([0.0], dtype=np.float32)

    signal = downsample(signal, sr, TARGET_SR)
    signal = trim_silence(signal)
    signal = normalize_audio(signal)
    signal = force_1d_clean(signal)

    return signal


# =======================
# 1. Upload database
# =======================
print("--- UPLOAD DATABASE WAV ---")
uploaded_db = files.upload()

db_audio = {}

for filename, content in uploaded_db.items():
    print(f"Memproses: {filename}")
    processed = process_audio_no_librosa(content)
    processed = force_1d_clean(processed)
    db_audio[filename] = processed
    print("   shape:", processed.shape)

print("\n✅ Semua file database selesai diproses.\n")


# =======================
# 2. Upload target
# =======================
print("--- UPLOAD FILE TARGET ---")
uploaded_target = files.upload()

results = []

print("\n--- MULAI PERHITUNGAN DTW ---")


# =======================
# 3. Perhitungan DTW
# =======================
for target_name, file_bytes in uploaded_target.items():

    target_signal = process_audio_no_librosa(file_bytes)
    target_signal = force_1d_clean(target_signal)

    print(f"\n▶ Menguji file: {target_name} | shape: {target_signal.shape}")

    for db_name, db_signal in db_audio.items():

        ts = force_1d_clean(target_signal)
        ds = force_1d_clean(db_signal)

        # Pastikan min panjang
        if len(ts) < 5:
            ts = np.pad(ts, (0, 5 - len(ts)))
        if len(ds) < 5:
            ds = np.pad(ds, (0, 5 - len(ds)))

        # Debug (jika perlu)
        # print("TS:", ts[:5], "DS:", ds[:5])

        # DTW aman
        distance, path = fastdtw(ts, ds, radius=10, dist=euclidean)

        print(f"   - {target_name} vs {db_name} → Jarak: {distance:.2f}")

        results.append({
            "File_Baru": target_name,
            "File_Database": db_name,
            "Jarak_DTW": distance
        })


# =======================
# 4. Simpan hasil
# =======================
df_results = pd.DataFrame(results).sort_values(by="Jarak_DTW")

print("\n--- 10 HASIL PALING MIRIP ---")
print(df_results.head(10))

df_results.to_csv("hasil_jarak_tutup_baru.csv", index=False)
print("\n✅ CSV disimpan: hasil_jarak_tutup_baru.csv")


--- UPLOAD DATABASE WAV ---


Saving Tutup.68egkmcg.ingestion-6d56d7dc5b-5ffk6.wav to Tutup.68egkmcg.ingestion-6d56d7dc5b-5ffk6 (5).wav
Saving Tutup.68egkoa9.ingestion-6d56d7dc5b-4mhbr.wav to Tutup.68egkoa9.ingestion-6d56d7dc5b-4mhbr (5).wav
Saving Tutup.68egkrrr.ingestion-6d56d7dc5b-5ffk6.wav to Tutup.68egkrrr.ingestion-6d56d7dc5b-5ffk6 (5).wav
Saving Tutup.68egkvct.ingestion-6d56d7dc5b-5ffk6.wav to Tutup.68egkvct.ingestion-6d56d7dc5b-5ffk6 (7).wav
Saving Tutup.68egl2kp.ingestion-6d56d7dc5b-4mhbr.wav to Tutup.68egl2kp.ingestion-6d56d7dc5b-4mhbr (6).wav
Saving Tutup.68egl9cu.ingestion-6d56d7dc5b-4nxks.wav to Tutup.68egl9cu.ingestion-6d56d7dc5b-4nxks (5).wav
Saving Tutup.68egl646.ingestion-6d56d7dc5b-5ffk6.wav to Tutup.68egl646.ingestion-6d56d7dc5b-5ffk6 (5).wav
Saving Tutup.68eglcql.ingestion-6d56d7dc5b-4mhbr.wav to Tutup.68eglcql.ingestion-6d56d7dc5b-4mhbr (5).wav
Saving Tutup.68eglg5g.ingestion-6d56d7dc5b-4nxks.wav to Tutup.68eglg5g.ingestion-6d56d7dc5b-4nxks (5).wav
Saving Tutup.68eglja2.ingestion-6d56d7dc5b-4mh

/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF prematurely; finished at 57388 bytes, expected 57396 bytes from header.
  sr, data = wavfile.read(io.BytesIO(file_bytes))
/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF prematurely; finished at 60118 bytes, expected 60126 bytes from header.
  sr, data = wavfile.read(io.BytesIO(file_bytes))
/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF prematurely; finished at 54658 bytes, expected 54666 bytes from header.
  sr, data = wavfile.read(io.BytesIO(file_bytes))
/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF prematurely; finished at 62850 bytes, expected 62858 bytes from header.
  sr, data = wavfile.read(io.BytesIO(file_bytes))
/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF prematurely; finished at 64044 bytes, expected 64052 bytes from header.
  sr, data = wavfile.read(io.BytesIO(file_bytes))
/tmp/ipython-input-2051587741.py:34: WavFileWarning: Reached EOF 

Saving Tutup.68egmfs5.ingestion-6d56d7dc5b-5ffk6.wav to Tutup.68egmfs5.ingestion-6d56d7dc5b-5ffk6 (6).wav

--- MULAI PERHITUNGAN DTW ---

▶ Menguji file: Tutup.68egmfs5.ingestion-6d56d7dc5b-5ffk6 (6).wav | shape: (2000,)


ValueError: Input vector should be 1-D.